In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from pathlib import Path
import os
from os.path import join
import sys
import sqlite3
# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully
EU1_PROD_Conn created successfully
EU2_PROD_Conn created successfully


In [3]:
LOG_PATH = os.path.join(INGESTER_LOG_PATH, 'SurveySummaryIngester.log')
Logger = Loggers(logger_name = 'SurveySummaryIngester', keys = ['File', 'Slack'])
Logger.clear_handlers()
import logging

file_handler = logging.FileHandler(LOG_PATH)
# Set date format to dd-mm-yyyy in log output
formatter = logging.Formatter('%(asctime)s [%(levelname)s] %(name)s: %(message)s', datefmt='%d-%m-%Y')
file_handler.setFormatter(formatter)
Logger.File.addHandler(file_handler)

slack_handler = logging.StreamHandler(SlackWriter(channel = 'C0B9PGDNHH7'))
Logger.Slack.addHandler(slack_handler)

In [4]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
KPI_META_KEYS = [
        'ReportAssetLengthKm',
        'AssetCoveredLengthKm',
        'DistributionPipeKm',
        'DistributionPipeCoveredKm',
        'CumulativeAssetCoveredLengthKm',
        'ServicePipeKm',
        'ServicePipeCoveredKm',
        'ReportCount',
        'DaysCount',
        'FOVMain',
        'SurveyDurationHours',
        'TargetDurationHours',
        'CustomerUtilization',
        'StarndardUtilization',
        'TotalSurveyors',
        'ProductivityPerSurveyor',
        'SurveyCount',
        'AvgSpeedKm',
        'SurveysCarDay',
        'IdleTime',
        'TotalDrivenLengthKm',
        'DrivingRatio',
        'NightDrivenLength',
        'DayDrivenLength',
        'NightRatio',
        'DayRatio',
        'PeakAboveSATCount',
        'LisaCount',
        'EmissionRate',
        'B0Count',
        'B1Count',
        'Bm1Count',
        'Bm2Count',
        'NGCount',
        'PGCount',
        'Not_NGCount',
        'LisaDensity',
        'InstatanoeusEmission',
        'B0Density',
        'B1Density',
        'Bm1Density',
        'Bm2Density',
        'B0Share',
        'B1Share',
        'Bm1Share',
        'Bm2Share',
        'NGShare',
        'PGShare',
        'Not_NGShare',
        'POR',
        'CurrentCompletion'
    ]


In [5]:
for view_name in ("Weekly_KPI_Plant", "Weekly_KPI", "Weekly_KPI_Region"):
    cursor.execute(f"DROP VIEW IF EXISTS {view_name};")
conn.commit()

In [6]:
query = """
CREATE VIEW Weekly_KPI_Plant AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    kd.BoundaryPlant,
    """ + ",\n    ".join([f"SUM(CASE WHEN kd.KPIId = '{kpi}' THEN kd.Value END) AS [{kpi}]" for kpi in KPI_META_KEYS]) + """
FROM KPI_Data kd
LEFT JOIN KPI_Customer kc ON kd.CustomerId = kc.CustomerId
WHERE kd.PeriodType = 'Week'
  AND kd.BoundaryPlant IS NOT NULL
  AND kd.BoundaryRegion IS NOT NULL
GROUP BY kd.Year, kd.PeriodValue, kd.CustomerId, kc.Name, kd.BoundaryRegion, kd.BoundaryPlant;"""
cursor.execute(query)
conn.commit()
print(query)


CREATE VIEW Weekly_KPI_Plant AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    kd.BoundaryPlant,
    SUM(CASE WHEN kd.KPIId = 'ReportAssetLengthKm' THEN kd.Value END) AS [ReportAssetLengthKm],
    SUM(CASE WHEN kd.KPIId = 'AssetCoveredLengthKm' THEN kd.Value END) AS [AssetCoveredLengthKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeKm' THEN kd.Value END) AS [DistributionPipeKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeCoveredKm' THEN kd.Value END) AS [DistributionPipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'CumulativeAssetCoveredLengthKm' THEN kd.Value END) AS [CumulativeAssetCoveredLengthKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeKm' THEN kd.Value END) AS [ServicePipeKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeCoveredKm' THEN kd.Value END) AS [ServicePipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'ReportCount' THEN kd.Value END) AS [ReportCount],
    SUM(CASE WHEN kd.KPIId = 'DaysCount' THEN kd.Value END) AS [DaysCount],
 

In [7]:
query = """
CREATE VIEW Weekly_KPI_Region AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    kd.BoundaryPlant,
    """ + ",\n    ".join([f"SUM(CASE WHEN kd.KPIId = '{kpi}' THEN kd.Value END) AS [{kpi}]" for kpi in KPI_META_KEYS]) + """
FROM KPI_Data kd
LEFT JOIN KPI_Customer kc ON kd.CustomerId = kc.CustomerId
WHERE kd.PeriodType = 'Week'
  AND kd.BoundaryRegion IS NOT NULL
  AND kd.BoundaryPlant IS NULL
GROUP BY kd.Year, kd.PeriodValue, kd.CustomerId, kc.Name, kd.BoundaryRegion, kd.BoundaryPlant;"""
cursor.execute(query)
conn.commit()
print(query)


CREATE VIEW Weekly_KPI_Region AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    kd.BoundaryPlant,
    SUM(CASE WHEN kd.KPIId = 'ReportAssetLengthKm' THEN kd.Value END) AS [ReportAssetLengthKm],
    SUM(CASE WHEN kd.KPIId = 'AssetCoveredLengthKm' THEN kd.Value END) AS [AssetCoveredLengthKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeKm' THEN kd.Value END) AS [DistributionPipeKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeCoveredKm' THEN kd.Value END) AS [DistributionPipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'CumulativeAssetCoveredLengthKm' THEN kd.Value END) AS [CumulativeAssetCoveredLengthKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeKm' THEN kd.Value END) AS [ServicePipeKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeCoveredKm' THEN kd.Value END) AS [ServicePipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'ReportCount' THEN kd.Value END) AS [ReportCount],
    SUM(CASE WHEN kd.KPIId = 'DaysCount' THEN kd.Value END) AS [DaysCount],


In [8]:
query = """
CREATE VIEW Weekly_KPI AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    """ + ",\n    ".join([f"SUM(CASE WHEN kd.KPIId = '{kpi}' THEN kd.Value END) AS [{kpi}]" for kpi in KPI_META_KEYS]) + """
FROM KPI_Data kd
LEFT JOIN KPI_Customer kc ON kd.CustomerId = kc.CustomerId
WHERE kd.PeriodType = 'Week'
  AND kd.BoundaryPlant IS NULL
  AND kd.BoundaryRegion IS NULL
GROUP BY kd.Year, kd.PeriodValue, kd.CustomerId, kc.Name, kd.BoundaryRegion;"""
cursor.execute(query)
conn.commit()

In [9]:
print(query)


CREATE VIEW Weekly_KPI AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    SUM(CASE WHEN kd.KPIId = 'ReportAssetLengthKm' THEN kd.Value END) AS [ReportAssetLengthKm],
    SUM(CASE WHEN kd.KPIId = 'AssetCoveredLengthKm' THEN kd.Value END) AS [AssetCoveredLengthKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeKm' THEN kd.Value END) AS [DistributionPipeKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeCoveredKm' THEN kd.Value END) AS [DistributionPipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'CumulativeAssetCoveredLengthKm' THEN kd.Value END) AS [CumulativeAssetCoveredLengthKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeKm' THEN kd.Value END) AS [ServicePipeKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeCoveredKm' THEN kd.Value END) AS [ServicePipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'ReportCount' THEN kd.Value END) AS [ReportCount],
    SUM(CASE WHEN kd.KPIId = 'DaysCount' THEN kd.Value END) AS [DaysCount],
    SUM(CASE WHEN kd.KPIId = 

In [13]:
Query(f"SELECT * FROM Weekly_KPI_Plant WHERE CustomerName = 'Cadent';").execute(KPIHub_Conn)

,Year,PeriodValue,CustomerName,BoundaryRegion,BoundaryPlant,ReportAssetLengthKm,AssetCoveredLengthKm,DistributionPipeKm,DistributionPipeCoveredKm,CumulativeAssetCoveredLengthKm,...,Bm2Density,B0Share,B1Share,Bm1Share,Bm2Share,NGShare,PGShare,Not_NGShare,POR,CurrentCompletion
0,2025,1,Cadent,North London,Wembley,18.67,16.23,18.16,15.85,None,...,0.00,17.86,3.57,78.57,0.00,85.71,14.29,0.00,None,None
1,2025,2,Cadent,North London,Rayleigh,135.50,121.01,131.12,118.00,None,...,0.12,25.45,2.42,63.64,8.48,87.57,10.06,2.37,None,None
2,2025,2,Cadent,North London,Slough,148.04,138.05,141.32,133.34,None,...,0.08,43.37,9.06,44.01,3.56,79.94,13.98,6.08,None,None
3,2025,2,Cadent,North London,Woodford,90.33,87.84,89.42,87.08,None,...,0.03,29.45,0.61,68.10,1.84,80.57,12.57,6.86,None,None
4,2025,3,Cadent,North London,Rayleigh,132.27,122.37,129.01,119.79,None,...,0.22,30.99,4.15,56.55,8.31,88.58,8.02,3.40,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
780,2026,37,Cadent,North London,Slough,31.49,30.01,30.73,29.40,None,...,0.10,37.50,0.00,58.33,4.17,83.78,13.51,2.70,None,None
781,2026,37,Cadent,North London,Woodford,NaN,NaN,NaN,NaN,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None
782,2026,37,Cadent,North West,Stockport,60.47,55.15,58.52,54.08,None,...,0.09,26.67,5.56,62.22,5.56,76.19,9.52,14.29,None,None
783,2026,37,Cadent,West Midlands,Birmingham,56.81,51.16,54.59,49.50,None,...,0.12,22.73,1.82,70.00,5.45,87.18,6.84,5.98,None,None
